## 1. PyTorch и нейронные сети (Устройство нейронных сетей)

### 1.1. Что такое PyTorch и для чего он нужен?
**PyTorch** — это библиотека с открытым исходным кодом для машинного обучения и глубокого обучения (Deep Learning), разработанная на базе фреймворка Torch (изначально Meta AI).

**Две главные возможности PyTorch:**
1. **Тензорные вычисления с ускорением на GPU (CUDA):** Тензор (`torch.Tensor`) — это многомерный массив (аналог `numpy.ndarray`), но с возможностью выполнять вычисления на видеокарте (GPU), что в десятки и сотни раз ускоряет работу с большими данными.
2. **Автоматическое дифференцирование (`Autograd`):** PyTorch автоматически строит вычислительный граф операций и вычисляет градиенты (производные), необходимые для обучения нейросетей методом обратного распространения ошибки (backpropagation).

---

### 1.2. Что такое нейронная сеть (NN)?
**Нейронная сеть (Neural Network, NN)** — это математическая модель, состоящая из последовательно или графово соединённых вычислительных единиц (**нейронов**), организованных в **слои**.

* **Входной слой (Input layer):** принимает исходные признаки объекта (например, пиксели изображения или столбцы таблицы).
* **Скрытые слои (Hidden layers):** выполняют линейные преобразования и применяют нелинейные **функции активации** (например, `ReLU`, `Sigmoid`, `Tanh`), позволяя сети находить сложные зависимости в данных.
* **Выходной слой (Output layer):** выдаёт итоговое предсказание модели (число в задаче регрессии или оценки классов в задаче классификации).

Каждый слой хранит **обучаемые параметры**:
* **Веса ($W$)** — коэффициенты важности входных связей.
* **Смещения ($b$, bias)** — сдвиг активации нейрона.

---

### 1.3. `torch.nn`, `nn.Module`, `nn.Linear`

| Компонент | Назначение и описание |
| :--- | :--- |
| **`torch.nn`** | Основной модуль PyTorch для построения нейронных сетей. Содержит готовые строительные блоки: слои (`Linear`, `Conv2d`), функции активации (`ReLU`, `Softmax`), функции потерь (`MSELoss`, `CrossEntropyLoss`) и контейнеры (`Sequential`). |
| **`nn.Module`** | Базовый класс для всех нейронных сетей и слоёв в PyTorch. Любая пользовательская архитектура наследуется от `nn.Module`. Он автоматически отслеживает все вложенные слои и их обучаемые параметры (`model.parameters()`), умеет переключать режимы (`model.train()` / `model.eval()`) и переносить модель на GPU (`.to('cuda')`). |
| **`nn.Linear(in_features, out_features)`** | Полносвязный (линейный) слой. Применяет линейное преобразование ко входным данным по формуле: $$y = xW^T + b$$ где `in_features` — размер входного вектора, а `out_features` — количество нейронов на выходе. |

---

### 1.4. Как работает метод `forward()`?
Метод **`forward(self, x)`** определяет **прямой проход (forward pass)** данных через нейронную сеть — то есть последовательность операций, которые преобразуют входной тензор `x` в выходной результат.

* В `__init__()` мы **объявляем** слои (создаём объекты `nn.Linear`, `nn.ReLU` и т.д.).
* В `forward(x)` мы **соединяем** их и описываем логику потока данных.
* **Важно:** При вызове модели на данных мы пишем `output = model(x)`, а **не** `model.forward(x)`. Вызов `model(x)` запускает внутренний метод `__call__` класса `nn.Module`, который выполняет служебные хуки (hooks) PyTorch для построения графа градиентов, а затем сам вызывает `forward(x)`.

In [2]:
import torch
import torch.nn as nn

# Пример создания простой нейронной сети
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) # Первый линейный слой
        self.relu = nn.ReLU()                         # Функция активации
        self.fc2 = nn.Linear(hidden_size, num_classes)# Выходной слой

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = SimpleNN(input_size=10, hidden_size=32, num_classes=3)
sample_input = torch.randn(5, 10)  # Батч из 5 объектов, по 10 признаков
logits = model(sample_input)       # Прямой проход (вызывает forward)
print("Форма выхода (логиты):", logits.shape)

Форма выхода (логиты): torch.Size([5, 3])


---
## 2. Функции потерь (Оценка ошибки модели)

### 2.1. Что такое функция потерь и зачем она нужна?
**Функция потерь (Loss Function / Criterion)** — это математическая функция, которая измеряет, насколько сильно предсказание модели ($\hat{y}$) расходится с истинным значением ($y$).

* Возвращает одно скалярное число — **значение ошибки (loss)**.
* Чем **меньше** значение функции потерь, тем **точнее** работает модель.

---

### 2.2. Основные функции потерь: `MSELoss`, `BCELoss`, `CrossEntropyLoss`

| Функция потерь | Задача | Что принимает на вход | Суть и формула |
| :--- | :--- | :--- | :--- |
| **`nn.MSELoss`** *(Mean Squared Error)* | **Регрессия** (предсказание непрерывного числа: цена, температура) | Непрерывные числа $\hat{y}$ и истинные значения $y$ | Средний квадрат разности между предсказанием и реальностью. Сильно штрафует за большие выбросы: $$L = rac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$ |
| **`nn.BCELoss`** *(Binary Cross-Entropy)* | **Бинарная классификация** (2 класса: 0 или 1) | Вероятности от `0` до `1` (после функции `Sigmoid`) и метки `0.0` или `1.0` | Измеряет расхождение между предсказанной вероятностью и бинарной меткой: $$L = -rac{1}{N} \sum_{i=1}^{N} [y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)]$$ *(На практике чаще используют `BCEWithLogitsLoss`, так как он объединяет `Sigmoid` и `BCELoss` для численной стабильности).* |
| **`nn.CrossEntropyLoss`** | **Многоклассовая классификация** ($K \ge 2$ классов) | **Сырые логиты** (без `Softmax`!) формы `(N, C)` и индексы правильных классов `0..C-1` формы `(N,)` | Внутри себя автоматически применяет `LogSoftmax` и `NLLLoss`. Штрафует модель, если вероятность правильного класса мала. |

---

### 2.3. Как оценивают ошибку модели?
1. Модель делает предсказание на обучающем батче (прямой проход).
2. Функция потерь сравнивает предсказания модели с правильными ответами (ground truth).
3. Если модель уверенно угадывает правильный ответ — ошибка близка к `0`. Если модель уверенно ошибается — значение `loss` резко возрастает (благодаря логарифму или возведению в квадрат).

---

### 2.4. Роль функции потерь в обучении
Функция потерь — это **компас (целевая функция)** для обучения нейросети:
* Обучение сводится к математической задаче **минимизации** функции потерь по всем параметрам сети ($W, b$).
* Именно от значения `loss` вычисляются градиенты (`loss.backward()`), которые показывают, в какую сторону и насколько сильно нужно изменить каждый вес, чтобы уменьшить ошибку на следующем шаге.

In [3]:
# Примеры использования функций потерь в PyTorch

# 1. MSELoss (Регрессия)
mse = nn.MSELoss()
y_pred_reg = torch.tensor([2.5, 0.0, 2.0])
y_true_reg = torch.tensor([3.0, -0.5, 2.0])
print("MSE Loss:", mse(y_pred_reg, y_true_reg).item())

# 2. BCELoss (Бинарная классификация — требует Sigmoid на входе)
bce = nn.BCELoss()
probs_bin = torch.sigmoid(torch.tensor([1.5, -2.0, 0.2])) # переводим логиты в вероятности
y_true_bin = torch.tensor([1.0, 0.0, 1.0])
print("BCE Loss:", bce(probs_bin, y_true_bin).item())

# 3. CrossEntropyLoss (Многоклассовая классификация — принимает СЫРЫЕ логиты)
ce = nn.CrossEntropyLoss()
raw_logits = torch.tensor([[2.0, 0.5, -1.0],   # объект 0 (больше всего баллов у класса 0)
                           [0.1, 2.5,  0.3]])  # объект 1 (больше всего баллов у класса 1)
y_true_cls = torch.tensor([0, 1])              # истинные индексы классов
print("CrossEntropy Loss:", ce(raw_logits, y_true_cls).item())

MSE Loss: 0.1666666716337204
BCE Loss: 0.3088267743587494
CrossEntropy Loss: 0.2124498039484024


---
## 3. Softmax и кросс-энтропия (Вероятности и классификация)

### 3.1. Что такое функция Softmax?
**Softmax** — это функция активации, применяемая на последнем слое нейронной сети в задачах многоклассовой классификации. Она преобразует вектор произвольных действительных чисел (**логитов**) в **распределение вероятностей** по классам.

**Свойства выхода Softmax:**
1. Каждое число находится в диапазоне от `0` до `1`.
2. Сумма всех значений по всем классам для одного объекта всегда строго равна `1` (100%).

---

### 3.2. Как из выходов получаются вероятности?
Пусть выходной слой сети выдал вектор сырых оценок (логитов) $z = [z_1, z_2, \dots, z_K]$ для $K$ классов. Функция Softmax вычисляет вероятность $i$-го класса по формуле:

$$	ext{Softmax}(z_i) = rac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

**Как это работает по шагам:**
1. **Экспоненцирование ($e^{z_i}$):** превращает любые числа (включая отрицательные) в строго положительные и усиливает разницу между максимальным логитом и остальными.
2. **Нормализация ($\div \sum e^{z_j}$):** делит каждую экспоненту на сумму всех экспонент, чтобы итоговая сумма равнялась `1`.

---

### 3.3. Что такое кросс-энтропия?
**Кросс-энтропия (Cross-Entropy)** — это мера различия между двумя распределениями вероятностей: истинным распределением $y$ (где правильный класс имеет вероятность `1`, а остальные — `0`) и предсказанным моделью распределением $\hat{y}$:

$$H(y, \hat{y}) = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)$$

Поскольку в one-hot векторе истинных меток $y_i = 1$ только для правильного класса $c$ (а для остальных $y_i = 0$), формула упрощается до отрицательного логарифма вероятности правильного класса:

$$L = -\log(\hat{y}_c)$$

* Если модель предсказала правильный класс с вероятностью $\hat{y}_c = 0.99$, то $-\log(0.99) pprox 0.01$ (ошибка почти ноль).
* Если модель предсказала правильный класс с вероятностью $\hat{y}_c = 0.01$, то $-\log(0.01) pprox 4.60$ (большой штраф).

---

### 3.4. Как связаны Softmax и кросс-энтропия?
1. **Логическая связка:** Softmax переводит сырые логиты сети в вероятности классов, а кросс-энтропия оценивает, насколько близки эти вероятности к истинному ответу.
2. **Практическая особенность в PyTorch (`nn.CrossEntropyLoss`):**
   * В PyTorch класс `nn.CrossEntropyLoss()` **уже включает в себя `Softmax`** (точнее, `LogSoftmax` + `NLLLoss`).
   * **Частая ошибка новичков:** поставить `nn.Softmax()` последним слоем модели и одновременно использовать `nn.CrossEntropyLoss()`. Это приведёт к двойному применению Softmax, затуханию градиентов и плохому обучению!
   * **Правило:** при обучении с `nn.CrossEntropyLoss()` модель должна возвращать **сырые логиты**. Функцию `torch.softmax(logits, dim=1)` применяют вручную только на этапе предсказания (инференса), когда нужно вывести человеку понятные проценты вероятностей.

In [4]:
# Демонстрация работы Softmax и связи с CrossEntropyLoss
logits = torch.tensor([[2.0, 1.0, 0.1]]) # Сырые выходы последнего линейного слоя

# Переводим логиты в вероятности вручную
probabilities = torch.softmax(logits, dim=1)
print("Сырые логиты:", logits)
print("Вероятности (Softmax):", probabilities)
print("Сумма вероятностей:", probabilities.sum().item())

# Проверяем связь с CrossEntropyLoss (пусть правильный класс = 0)
target = torch.tensor([0])
loss_pytorch = nn.CrossEntropyLoss()(logits, target)
loss_manual = -torch.log(probabilities[0, 0]) # -log(p_correct)

print(f"PyTorch CrossEntropyLoss: {loss_pytorch.item():.4f}")
print(f"Ручной расчёт -log(p):    {loss_manual.item():.4f}")

Сырые логиты: tensor([[2.0000, 1.0000, 0.1000]])
Вероятности (Softmax): tensor([[0.6590, 0.2424, 0.0986]])
Сумма вероятностей: 1.0000001192092896
PyTorch CrossEntropyLoss: 0.4170
Ручной расчёт -log(p):    0.4170


---
## 4. Оптимизаторы (Обновление параметров модели)

### 4.1. Что такое оптимизатор и градиентный спуск?
* **Градиентный спуск (Gradient Descent)** — это итерационный алгоритм поиска минимума функции потерь. Градиент ($
abla L$) указывает направление наискорейшего **роста** функции, поэтому для уменьшения ошибки мы двигаемся в сторону **антиградиента** (противоположном направлении):
  $$w_{	ext{new}} = w_{	ext{old}} - \eta \cdot rac{\partial L}{\partial w}$$
* **Оптимизатор (`torch.optim`)** — это объект в PyTorch, который хранит ссылки на все обучаемые параметры модели (`model.parameters()`) и реализует конкретный алгоритм обновления этих весов на основе вычисленных градиентов.

---

### 4.2. Оптимизаторы `SGD` и `Adam`

| Характеристика | `torch.optim.SGD` *(Stochastic Gradient Descent)* | `torch.optim.Adam` *(Adaptive Moment Estimation)* |
| :--- | :--- | :--- |
| **Принцип работы** | Обновляет веса в направлении антиградиента, вычисленного по мини-батчу данных. Часто используется с моментом (`momentum`), который накапливает «инерцию» движения. | Адаптивный алгоритм. Объединяет идеи **Momentum** (учёт прошлых градиентов) и **RMSProp** (индивидуальная настройка шага для каждого веса на основе квадратов градиентов). |
| **Скорость обучения** | Единый `lr` для всех параметров сети. | Автоматически адаптирует эффективный шаг обучения для каждого отдельного параметра. |
| **Плюсы и применение** | Простой, требует меньше памяти, часто даёт лучшую обобщающую способность в задачах компьютерного зрения при тонкой настройке расписания `lr`. | Быстро сходится, устойчив к выбору начального `lr`, работает «из коробки». **Стандарт по умолчанию** для большинства задач. |

---

### 4.3. Скорость обучения (`learning rate`, `lr`)
**Learning rate ($\eta$)** — это гиперпараметр, определяющий размер шага, с которым оптимизатор обновляет веса модели в направлении антиградиента.

* **Слишком большой `lr`:** шаги слишком огромные, модель «перепрыгивает» минимум функции потерь; значение `loss` скачет или уходит в бесконечность (`NaN`).
* **Слишком маленький `lr`:** обучение идёт крайне медленно, требуется огромное количество эпох, модель может застрять в плохом локальном минимуме.
* **Типичные стартовые значения:** для `Adam` обычно берут `1e-3` (`0.001`) или `3e-4`, для `SGD` — `1e-2` (`0.01`) или `1e-1` (`0.1`).

---

### 4.4. Цикл обучения: `optimizer.zero_grad()`, `loss.backward()` и `optimizer.step()`
Каждый шаг обучения (обучающая итерация на одном батче) в PyTorch строго состоит из трёх обязательных команд:

1. **`optimizer.zero_grad()`** — **обнуляет** градиенты с предыдущего шага. По умолчанию PyTorch *накапливает* (суммирует) градиенты в атрибуте `.grad` каждого тензора при каждом вызове `.backward()`. Без обнуления старые градиенты смешаются с новыми.
2. **`loss.backward()`** — запускает алгоритм **обратного распространения ошибки (backpropagation)**. PyTorch проходит по вычислительному графу от `loss` к входным слоям и вычисляет градиенты функции потерь по каждому параметру (`param.grad`).
3. **`optimizer.step()`** — **обновляет веса** модели (`W` и `b`), используя вычисленные на шаге 2 градиенты (`param.grad`) и выбранную стратегию оптимизатора (`SGD`, `Adam` и др.).

In [5]:
# Полный минимальный цикл обучения в PyTorch (10 шагов)
torch.manual_seed(42)

# 1. Данные: 20 объектов, 4 признака, 2 класса
X = torch.randn(20, 4)
y = torch.randint(0, 2, (20,))

# 2. Модель, функция потерь и оптимизатор
model = SimpleNN(input_size=4, hidden_size=16, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 3. Цикл обучения
for epoch in range(1, 11):
    # Шаг 1: обнуляем старые градиенты
    optimizer.zero_grad()

    # Прямой проход: получаем логиты
    outputs = model(X)

    # Считаем ошибку
    loss = criterion(outputs, y)

    # Шаг 2: обратный проход (вычисляем градиенты)
    loss.backward()

    # Шаг 3: обновляем веса модели
    optimizer.step()

    if epoch % 2 == 0 or epoch == 1:
        print(f"Эпоха {epoch:2d} | Loss: {loss.item():.4f}")

Эпоха  1 | Loss: 0.7013
Эпоха  2 | Loss: 0.6813
Эпоха  4 | Loss: 0.6491
Эпоха  6 | Loss: 0.6232
Эпоха  8 | Loss: 0.6001
Эпоха 10 | Loss: 0.5776


---
## Связь всех понятий

1. **Как все 4 темы работают вместе в едином конвейере?**
   * **Модель (`nn.Module`)** принимает входные данные `x` и через метод **`forward()`** пропускает их через слои (`nn.Linear`), выдавая сырые оценки — **логиты**.
   * **Функция потерь (`Loss`)** сравнивает эти предсказания с правильными ответами и вычисляет числовую ошибку. (Для многоклассовой классификации `CrossEntropyLoss` сама внутри применяет **`Softmax`** к логитам, чтобы получить вероятности, и считает кросс-энтропию).
   * Далее вызывается **`loss.backward()`**, который вычисляет градиенты ошибки по всем параметрам модели.
   * **Оптимизатор (`SGD` / `Adam`)** через **`optimizer.step()`** сдвигает веса модели в сторону уменьшения ошибки на величину, пропорциональную **`learning rate`**.

2. **Главные отличия, о которых часто спрашивают:**
   * **`nn.MSELoss` vs `nn.CrossEntropyLoss`:** первая — для регрессии (непрерывные числа), вторая — для классификации (дискретные классы).
   * **`nn.BCELoss` vs `nn.CrossEntropyLoss`:** `BCELoss` — для 2 классов и требует `Sigmoid` на входе; `CrossEntropyLoss` — для $K$ классов и требует **сырые логиты** (без `Softmax`).
   * **`loss.backward()` vs `optimizer.step()`:** `backward()` только *вычисляет* градиенты и записывает их в `.grad`, не меняя веса; `step()` непосредственно *изменяет* значения весов на основе этих градиентов.
   * **`SGD` vs `Adam`:** `SGD` использует единый `learning rate` для всех весов, а `Adam` адаптирует `learning rate` индивидуально для каждого параметра и использует инерцию (момент).